<a href="https://colab.research.google.com/github/keivernunez/dataminingavanzado_austral/blob/main/Clase2_Note3_de_4_SVM_%2B_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Máquinas de Vectores de Soporte (SVM) con Aceleración por GPU

**Introducción:**

Este cuaderno es una versión optimizada del análisis de Máquinas de Vectores de Soporte, adaptado para su ejecución en **Google Colab con una instancia de GPU**. El objetivo principal es demostrar cómo el uso de librerías aceleradas por hardware, como RAPIDS cuML, puede reducir drásticamente el tiempo de entrenamiento de los modelos en comparación con las implementaciones tradicionales basadas en CPU, como la de Scikit-learn.

**Requisito Previo:** Para ejecutar este cuaderno correctamente, es **indispensable** cambiar el entorno de ejecución de Colab a una instancia con GPU. Esto se realiza en el menú `Entorno de ejecución -> Cambiar tipo de entorno de ejecución` y seleccionando `GPU` como acelerador de hardware.

Se abordarán los siguientes puntos:
1.  Instalación de las librerías de RAPIDS.
2.  Comparativa de tiempos de entrenamiento entre `cuml.svm.SVC` (GPU) y `sklearn.svm.SVC` (CPU).
3.  Ejemplo completo de clasificación con el dataset del Titanic.
4.  Visualización de fronteras de decisión y análisis de clustering.

## 1. Instalación de RAPIDS cuML

El primer paso es instalar la librería `cuml-cu11` que contiene la implementación de algoritmos de machine learning acelerados por GPU y es compatible con las GPUs disponibles en Google Colab. Este proceso puede tardar varios minutos.

In [ ]:
# Detectar la versión de CUDA y resolver conflictos de CuPy antes de instalar RAPIDS
import subprocess
import re

# Obtener versión de CUDA
result = subprocess.run(['nvcc', '--version'], stdout=subprocess.PIPE, text=True)
output = result.stdout

# Extraer la versión de lanzamiento de CUDA (e.g., 12.3)
cuda_version_match = re.search(r'release (\d+\.\d+)', output)
if cuda_version_match:
    cuda_full_version = cuda_version_match.group(1)
    cuda_major = cuda_full_version.split('.')[0]
    if cuda_major == '11':
        cuda_suffix = 'cu11'
    elif cuda_major == '12':
        cuda_suffix = 'cu12'
    else:
        raise ValueError(f"Versión mayor de CUDA no soportada: {cuda_major}")
else:
    raise ValueError("No se pudo detectar la versión de CUDA")

print(f"Versión de CUDA detectada: {cuda_full_version}. Usando sufijo: {cuda_suffix}")

# Desinstalar paquetes CuPy conflictivos
!pip uninstall -y cupy-cuda11x cupy-cuda12x cupy-cuda118 cupy-cuda117 cupy-cuda116 cupy-cuda115 cupy-cuda114 cupy-cuda113 cupy-cuda112 cupy-cuda111 cupy-cuda110 cupy-cuda102 cupy-cuda101 cupy-cuda100 cupy-cuda92 cupy-cuda91 cupy-cuda90 cupy

# Instalar cuda-python si falta
!pip install cuda-python

# Instalar cuDF y cuML apropiados
!pip install --extra-index-url=https://pypi.nvidia.com cudf-{cuda_suffix} cuml-{cuda_suffix} -q

# Nota: Después de ejecutar esto, puede que necesites reiniciar el runtime para que los cambios surtan efecto

## 2. Configuración del Entorno y Carga de Datos

Se importan las librerías necesarias, incluyendo `cuml`, `cupy` para el manejo de arrays en GPU, y las librerías estándar para manipulación de datos y visualización.

In [ ]:
# Importaciones para GPU y CPU
import numpy as np
import pandas as pd
import cupy
import cudf
import cuml
import time

# Importaciones estándar de Scikit-learn y visualización
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVC as SklearnSVC # Se renombra para evitar conflictos
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from sklearn.datasets import make_moons, load_iris
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Configuraciones generales
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
# Carga y preparación del dataset Titanic (en CPU con Pandas)
titanic = sns.load_dataset('titanic')
titanic = titanic[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']].copy()
titanic = titanic.dropna(subset=['embarked'])

# Definición de características y objetivo
X = titanic.drop(columns=['survived'])
y = titanic['survived']

# Identificación de tipos de columnas
num_cols = ['age', 'sibsp', 'parch', 'fare']
cat_cols = ['pclass', 'sex', 'embarked']

# División en train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

## 3. Preprocesamiento de Datos

Se define un pipeline de preprocesamiento utilizando Scikit-learn. Este pipeline se encargará de la imputación de valores faltantes, el escalado de características numéricas y la codificación de variables categóricas. El pipeline se ejecutará sobre la CPU y su salida (un array de NumPy) será transferida a la GPU para el entrenamiento.

In [ ]:
# Pipeline de preprocesamiento (se ejecuta en CPU)
num_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Aplicar el preprocesamiento a los datos
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

## 4. Comparativa de Rendimiento: GPU vs. CPU

A continuación, se entrenará un modelo SVM con los mismos hiperparámetros (`C=10`, `gamma='scale'`) tanto en la GPU con `cuml` como en la CPU con `scikit-learn`. Se medirán los tiempos de entrenamiento para cuantificar la ganancia en velocidad.

In [ ]:
# --- Entrenamiento en GPU con cuML ---
print("Iniciando entrenamiento en GPU...")

# 1. Mover los datos procesados a la GPU usando cupy
X_train_gpu = cupy.asarray(X_train_processed)
y_train_gpu = cupy.asarray(y_train.values)
X_test_gpu = cupy.asarray(X_test_processed)

# 2. Instanciar el clasificador SVM de cuML
cuml_svc = cuml.svm.SVC(C=10, kernel='rbf', gamma='scale', probability=True, random_state=42)

# 3. Medir tiempo de entrenamiento
start_time_gpu = time.time()
cuml_svc.fit(X_train_gpu, y_train_gpu)
end_time_gpu = time.time()
gpu_training_time = end_time_gpu - start_time_gpu

print(f"Entrenamiento en GPU completado en {gpu_training_time:.4f} segundos.")

# --- Entrenamiento en CPU con Scikit-learn ---
print("\nIniciando entrenamiento en CPU...")

# 1. Instanciar el clasificador SVM de Scikit-learn
sklearn_svc = SklearnSVC(C=10, kernel='rbf', gamma='scale', probability=True, random_state=42)

# 2. Medir tiempo de entrenamiento (los datos ya están en memoria de CPU)
start_time_cpu = time.time()
sklearn_svc.fit(X_train_processed, y_train)
end_time_cpu = time.time()
cpu_training_time = end_time_cpu - start_time_cpu

print(f"Entrenamiento en CPU completado en {cpu_training_time:.4f} segundos.")

# --- Comparativa ---
print(f"\nSpeedup (aceleración) aproximado con GPU: {cpu_training_time / gpu_training_time:.2f}x")

## 5. Evaluación del Modelo Entrenado en GPU

Se evalúa el modelo de `cuML` en el conjunto de prueba. Para esto, las predicciones generadas en la GPU deben ser transferidas de vuelta a la memoria principal (CPU) para poder utilizar las funciones de métricas de `scikit-learn`.

In [ ]:
# Realizar predicciones en la GPU
y_pred_gpu = cuml_svc.predict(X_test_gpu)

# Mover las predicciones a la CPU para la evaluación
y_pred_cpu = cupy.asnumpy(y_pred_gpu)

# Evaluación del modelo
print(f'Exactitud (Accuracy) en prueba: {(y_pred_cpu == y_test).mean():.4f}')
print('\nReporte de Clasificación:\n', classification_report(y_test, y_pred_cpu))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred_cpu)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusión (Modelo GPU)')
plt.xlabel('Predicción')
plt.ylabel('Valor Real')
plt.show()

## 6. Visualización de Fronteras de Decisión

Se utiliza un conjunto de datos sintético para visualizar cómo los diferentes hiperparámetros (`C` y `kernel`) afectan la frontera de decisión del clasificador SVM. Este análisis se realiza con la versión de Scikit-learn por simplicidad en la visualización, pero los conceptos son idénticos para la implementación de cuML.

In [ ]:
# Creación de un dataset sintético
X_synth, y_synth = make_moons(n_samples=200, noise=0.2, random_state=42)
X_synth_scaled = StandardScaler().fit_transform(X_synth)

# Función para visualizar la frontera de decisión
def plot_decision_boundary(clf, X, y, title):
    h = .02
    x_min, x_max = X[:, 0].min() - .5, X[:, 0].max() + .5
    y_min, y_max = X[:, 1].min() - .5, X[:, 1].max() + .5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, cmap=plt.cm.coolwarm, alpha=0.8)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.coolwarm, edgecolors='k')
    plt.title(title)
    plt.show()

# Modelos con diferentes configuraciones
svm_rbf_low_c = SklearnSVC(kernel='rbf', C=0.1, gamma='auto').fit(X_synth_scaled, y_synth)
plot_decision_boundary(svm_rbf_low_c, X_synth_scaled, y_synth, 'SVM con Kernel RBF (C=0.1, Margen Suave)')

svm_rbf_high_c = SklearnSVC(kernel='rbf', C=100, gamma='auto').fit(X_synth_scaled, y_synth)
plot_decision_boundary(svm_rbf_high_c, X_synth_scaled, y_synth, 'SVM con Kernel RBF (C=100, Margen Duro)')

## 7. Análisis de Clustering (Ejemplo Adicional)

Se incluye un ejemplo de clustering con el dataset Iris para determinar el número óptimo de agrupaciones mediante el método del Codo y el puntaje de Silueta. Este proceso se ejecuta en CPU con Scikit-learn.

In [ ]:
# Carga del dataset Iris
iris = load_iris()
X_iris = iris.data

# Método del Codo
inertia = []
k_range = range(1, 11)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_iris)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 6))
plt.plot(k_range, inertia, marker='o')
plt.xlabel('Número de Clusters (k)')
plt.ylabel('Inercia')
plt.title('Método del Codo para Selección de k')
plt.show()

# Análisis de Silueta (para k=3)
best_k = 3
labels = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit_predict(X_iris)
silhouette_avg = silhouette_score(X_iris, labels)
print(f'Para k = {best_k}, el puntaje de silueta promedio es: {silhouette_avg:.4f}')

## Síntesis de lo visto en el Notebook

El uso de `cuML` para entrenar modelos SVM en una GPU puede ofrecer un **aumento de rendimiento (speedup) muy significativo**, especialmente en conjuntos de datos de tamaño mediano a grande. La sintaxis es notablemente similar a la de `scikit-learn`, lo que facilita la transición para desarrolladores familiarizados con ese ecosistema.

Es importante considerar que existe un costo computacional asociado a la transferencia de datos entre la memoria de la CPU y la GPU. Para datasets muy pequeños, el entrenamiento en CPU puede ser más rápido y práctico debido a este sobrecosto. Sin embargo, a medida que la complejidad y el tamaño de los datos aumentan, la aceleración por GPU se vuelve una herramienta indispensable para la experimentación ágil y el entrenamiento eficiente de modelos.